In [1]:
import os
import json
import pandas as pd

from pathlib import Path


# Get GEO_ID from folder name (same pattern as prepare_data.ipynb)
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))


description_path = Path('description.json')
preprocessed_csv_path = Path(f"{GEO_ID}_preprocessed.csv")

if not description_path.exists():
    raise FileNotFoundError(
        "description.json not found. Please run 'prepare_data.ipynb' first to generate it."
    )

if not preprocessed_csv_path.exists():
    raise FileNotFoundError(
        f"{preprocessed_csv_path} not found. Please run 'preprocess_data.ipynb' first to generate it."
    )

with open(description_path) as f:
    summary_data = json.load(f)

df = pd.read_csv(preprocessed_csv_path)

print("Data contains", df.shape[0], "samples and", df.shape[1], "columns.")
print("First few rows of the data:")
df.head()


Data contains 6 samples and 13528 columns.
First few rows of the data:


,sample_title,patient_id,tissue,diagnosis,glioblastoma subtype,brain location,g207 treatment,age (years),Sex,survival post g207 (days),...,COX2,ATP8,ATP6,COX3,ND3,ND4L,ND4,ND5,ND6,CYTB
0,Patient_1_post_G207,Patient_101,brain tumor,glioblastoma,Mesenchymal,left frontal,post-G207 inoculation,52,male,113,...,8.320274,6.762154,8.169685,8.294988,6.538236,7.240946,9.108596,8.386226,7.310538,8.257274
1,Patient_3_post_G207,Patient_103,brain tumor,glioblastoma,Classical/Proneural,left frontoparietal,post-G207 inoculation,57,female,233,...,7.968605,6.508388,7.827610,7.892162,5.862397,7.058654,8.854845,8.198946,6.976693,7.716466
2,Patient_5_post_G207,Patient_105,brain tumor,glioblastoma,Mesenchymal,right frontal,post-G207 inoculation,39,female,263,...,8.099085,6.804977,8.165538,8.067695,6.414345,7.159066,8.997143,8.503342,7.394330,7.956114
3,Patient_6_post_G207,Patient_106,brain tumor,glioblastoma,Mesenchymal,right frontoparietal,post-G207 inoculation,47,female,165,...,8.903723,7.382895,8.743258,8.590933,7.334513,7.837764,9.611611,8.810894,7.658829,8.936037
4,Patient_7_post_G207,Patient_107,brain tumor,glioblastoma,Mesenchymal,right frontal,post-G207 inoculation,66,female,623,...,8.557609,7.049055,8.461490,8.463640,6.848814,7.292998,9.292928,8.454921,7.487190,8.418266


In [2]:
n_stats_written = 0

for idx, endpoint in enumerate(summary_data['survival-endpoints']):
    time_var = endpoint['time_var']['var_name']
    event_var = endpoint['event_var']['var_name']

    if time_var not in df.columns or event_var not in df.columns:
        abbrv = endpoint.get('abbrv', f'endpoint[{idx}]')
        print(
            f"Skipping stats for {abbrv}: columns not in preprocessed table "
            f"({time_var!r}, {event_var!r})."
        )
        endpoint.pop('stats', None)
        continue

    complete_mask = df[[time_var, event_var]].notnull().all(axis=1)
    n_complete = int(complete_mask.sum())
    n_incomplete = int(len(df) - n_complete)

    # Among samples with complete time + event data
    censored_count = int((complete_mask & (df[event_var] == 0.0)).sum())
    event_count = int((complete_mask & (df[event_var] == 1.0)).sum())

    censored_ratio = censored_count / n_complete

    print(f"Number of samples with incomplete data for {time_var} or {event_var}: {n_incomplete}")
    print(f"Number of samples with complete data for both variables: {n_complete}")
    print(f"Number of censored samples (event_var == 0.0): {censored_count}")
    print(f"Number of events (event_var == 1.0): {event_count}")
    pct = 100.0 * censored_ratio
    print(
        f"Censored ratio (among complete): {censored_count} / {n_complete} = "
        f"{censored_ratio:.3f} ({pct:.1f}%)"
    )

    # Refresh only this block on each run; leave all other endpoint fields unchanged
    endpoint['stats'] = {
        'n_incomplete': n_incomplete,
        'n_complete': n_complete,
        'n_censored': censored_count,
        'n_events': event_count,
        'censored_ratio': round(censored_ratio, 2),
    }
    n_stats_written += 1

with open(description_path, 'w', encoding='utf-8') as f:
    json.dump(summary_data, f, indent=4, ensure_ascii=False)
    f.write('\n')

print(
    f"Updated stats for {n_stats_written} of {len(summary_data['survival-endpoints'])} endpoint(s); "
    f"wrote {description_path.resolve()}"
)


Skipping stats for OS: columns not in preprocessed table ('survival post g207 (days)', 'unknown').
Updated stats for 0 of 1 endpoint(s); wrote /Users/jakakokosar/dev/phd/survival-rnaseq-data/datasets/geo/GSE162643/description.json
